# Reaggregate 2024-25 data to 2025-26 statistical report questions

This notebook:
1. Compares the 2024-2025 vs 2025-2026 question sets directly from the XLSX templates.
2. Builds a crosswalk for normalized questions (exact and fuzzy matching).
3. Reaggregates prior long-form data into a 2025-26-aligned dataset.

Questions that cannot be normalized into a common format are still included with blank values for unmatched years.

In [ ]:
from __future__ import annotations

from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET
import re
from difflib import SequenceMatcher

import pandas as pd

In [ ]:
TEMPLATE_2024 = Path('2024-2025 Statistical Reports.xlsx')
TEMPLATE_2025 = Path('2025-26 Statistical Reports (1).xlsx')

# Prior long-form data (2024-25 schema), expected columns include at least:
# gc_orgID, institution_en, ReportingPeriodStart, ReportingPeriodEnd, id, value
PRIOR_LONG_DATA = Path('combined_form_data_long_2024_25.csv')

# Outputs
OUT_DIR = Path('.')
QUESTION_DIFF_OUT = OUT_DIR / 'ati_question_comparison_2024_25_vs_2025_26.csv'
CROSSWALK_OUT = OUT_DIR / 'ati_question_crosswalk_2024_25_to_2025_26.csv'
REALIGNED_OUT = OUT_DIR / 'combined_form_data_long_2025_26_reaggregated.csv'

In [ ]:
NS = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
      'rel': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
      'pkgrel': 'http://schemas.openxmlformats.org/package/2006/relationships'}

def _shared_strings(zf: zipfile.ZipFile) -> list[str]:
    try:
        root = ET.fromstring(zf.read('xl/sharedStrings.xml'))
    except KeyError:
        return []
    vals = []
    for si in root.findall('main:si', NS):
        parts = [t.text or '' for t in si.findall('.//main:t', NS)]
        vals.append(''.join(parts).strip())
    return vals

def _workbook_sheet_map(zf: zipfile.ZipFile) -> dict[str, str]:
    wb = ET.fromstring(zf.read('xl/workbook.xml'))
    rels = ET.fromstring(zf.read('xl/_rels/workbook.xml.rels'))
    rel_map = {r.attrib['Id']: r.attrib['Target'] for r in rels.findall('pkgrel:Relationship', NS)}
    out = {}
    for s in wb.findall('main:sheets/main:sheet', NS):
        rid = s.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
        target = rel_map[rid]
        if not target.startswith('worksheets/'):
            continue
        out[s.attrib['name']] = f'xl/{target}'
    return out

def _cell_value(c, shared):
    t = c.attrib.get('t')
    v = c.find('main:v', NS)
    if v is None or v.text is None:
        return ''
    raw = v.text
    if t == 's':
        return shared[int(raw)] if raw.isdigit() and int(raw) < len(shared) else ''
    return raw

def extract_questions(xlsx_path: Path, sheet='ATI/LAI') -> pd.DataFrame:
    with zipfile.ZipFile(xlsx_path) as zf:
        shared = _shared_strings(zf)
        smap = _workbook_sheet_map(zf)
        xml_path = smap.get(sheet)
        if not xml_path:
            raise KeyError(f'Sheet {sheet!r} not found in {xlsx_path.name}. Available: {list(smap)}')
        root = ET.fromstring(zf.read(xml_path))

    rows = []
    for row in root.findall('.//main:sheetData/main:row', NS):
        vals = {}
        for c in row.findall('main:c', NS):
            ref = c.attrib.get('r', '')
            col = re.sub(r'\d+', '', ref)
            vals[col] = _cell_value(c, shared).strip()

        qid = vals.get('A', '')
        label = vals.get('B', '')
        if re.match(r'^Q\d+', qid):
            rows.append({'id': qid, 'question_label': label})

    df = pd.DataFrame(rows).drop_duplicates(subset=['id']).reset_index(drop=True)
    return df

In [ ]:
q24 = extract_questions(TEMPLATE_2024)
q25 = extract_questions(TEMPLATE_2025)

# Simple normalization key for text matching
def norm_text(s: str) -> str:
    s = (s or '').lower()
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

q24['norm'] = q24['question_label'].map(norm_text)
q25['norm'] = q25['question_label'].map(norm_text)

In [ ]:
# Build crosswalk: exact id, exact normalized text, then fuzzy text
q25_by_id = dict(zip(q25['id'], q25['question_label']))
q25_by_norm = q25.groupby('norm', as_index=False).first()

rows = []
for _, r in q24.iterrows():
    old_id, old_label, old_norm = r['id'], r['question_label'], r['norm']

    match_type = 'unmatched'
    new_id = None
    new_label = None
    score = 0.0

    if old_id in q25_by_id:
        match_type = 'same_id'
        new_id = old_id
        new_label = q25_by_id[old_id]
        score = 1.0
    else:
        exact = q25_by_norm[q25_by_norm['norm'] == old_norm]
        if len(exact) == 1:
            match_type = 'exact_text'
            new_id = exact.iloc[0]['id']
            new_label = exact.iloc[0]['question_label']
            score = 1.0
        else:
            best = None
            for _, c in q25.iterrows():
                s = SequenceMatcher(None, old_norm, c['norm']).ratio()
                if best is None or s > best[0]:
                    best = (s, c['id'], c['question_label'])
            if best and best[0] >= 0.88:
                match_type = 'fuzzy_text'
                score, new_id, new_label = best

    rows.append({
        'old_id': old_id,
        'old_question_label': old_label,
        'new_id': new_id,
        'new_question_label': new_label,
        'match_type': match_type,
        'match_score': score,
    })

# Include 2025-26-only questions
mapped_new_ids = {r['new_id'] for r in rows if r['new_id']}
for _, r in q25.iterrows():
    if r['id'] not in mapped_new_ids:
        rows.append({
            'old_id': None,
            'old_question_label': None,
            'new_id': r['id'],
            'new_question_label': r['question_label'],
            'match_type': 'new_only',
            'match_score': 1.0,
        })

crosswalk = pd.DataFrame(rows).sort_values(['new_id','old_id'], na_position='last').reset_index(drop=True)
crosswalk.to_csv(CROSSWALK_OUT, index=False)

# Comparison table
comparison = crosswalk.copy()
comparison['status'] = comparison['match_type'].map({
    'same_id': 'common',
    'exact_text': 'common',
    'fuzzy_text': 'common_fuzzy',
    'unmatched': 'old_only',
    'new_only': 'new_only',
}).fillna('unknown')
comparison.to_csv(QUESTION_DIFF_OUT, index=False)

print(f'Wrote {CROSSWALK_OUT}')
print(f'Wrote {QUESTION_DIFF_OUT}')
print(comparison['status'].value_counts(dropna=False))

In [ ]:
if PRIOR_LONG_DATA.exists():
    prior = pd.read_csv(PRIOR_LONG_DATA)

    required = {'id', 'value'}
    missing = required - set(prior.columns)
    if missing:
        raise ValueError(f'Missing required columns in {PRIOR_LONG_DATA}: {missing}')

    # Build one-to-many map from old ids to 2025-26 ids
    id_map = crosswalk.dropna(subset=['old_id','new_id']).copy()
    id_map = id_map[id_map['match_type'].isin(['same_id','exact_text','fuzzy_text'])]

    remapped = prior.merge(id_map[['old_id','new_id','new_question_label']], left_on='id', right_on='old_id', how='left')
    remapped['id_2025_26'] = remapped['new_id']

    key_cols = [c for c in ['gc_orgID','institution_en','institution_fr','ReportingPeriodStart','ReportingPeriodEnd'] if c in remapped.columns]
    grouped = (
        remapped.dropna(subset=['id_2025_26'])
        .groupby(key_cols + ['id_2025_26'], dropna=False, as_index=False)['value']
        .sum()
        .rename(columns={'id_2025_26':'id'})
    )

    # Ensure all 2025-26 questions exist (blank where unavailable)
    universe = q25[['id','question_label']].drop_duplicates()
    if key_cols:
        entities = remapped[key_cols].drop_duplicates()
        entities['__k'] = 1
        universe2 = universe.copy(); universe2['__k'] = 1
        full = entities.merge(universe2, on='__k').drop(columns='__k')
        out = full.merge(grouped, on=key_cols+['id'], how='left')
    else:
        out = universe.merge(grouped, on=['id'], how='left')

    out['source_year'] = '2024-25'
    out['target_question_set'] = '2025-26'
    out.to_csv(REALIGNED_OUT, index=False)
    print(f'Wrote {REALIGNED_OUT} ({len(out):,} rows)')
else:
    print(f'Skipped reaggregation because {PRIOR_LONG_DATA} does not exist.')